# Formula 1 Exploratory Data Analysis (EDA)

This notebook combines **Structured Data Analysis (CSVs)** and **Unstructured Image Analysis (FastAI)** to explore the F1DB dataset.

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Supress warnings
import warnings
warnings.filterwarnings('ignore')

## 2. Structured Data Analysis (CSVs)

In [ ]:
# Paths to the CSVs in the F1 DB folder
CSV_DIR = '../f1 db'

drivers_df = pd.read_csv(f'{CSV_DIR}/drivers.csv', na_values='\\N')
races_df = pd.read_csv(f'{CSV_DIR}/races.csv', na_values='\\N')
results_df = pd.read_csv(f'{CSV_DIR}/results.csv', na_values='\\N')

print(f"Drivers: {drivers_df.shape[0]} rows")
print(f"Races: {races_df.shape[0]} rows")
print(f"Results: {results_df.shape[0]} rows")
drivers_df.head()

In [ ]:
# Null values check
print("Missing values in results:")
results_df.isnull().sum()

### Feature Engineering: Top 10 Drivers by Total Career Points

In [ ]:
results_df['points'] = pd.to_numeric(results_df['points'], errors='coerce').fillna(0)
driver_points = results_df.groupby('driverId')['points'].sum().reset_index()

# Merge with driver names
driver_stats = pd.merge(driver_points, drivers_df[['driverId', 'forename', 'surname']], on='driverId')
driver_stats['Full Name'] = driver_stats['forename'] + ' ' + driver_stats['surname']

top_10_drivers = driver_stats.sort_values(by='points', ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x='points', y='Full Name', data=top_10_drivers, palette='viridis')
plt.title('Top 10 F1 Drivers by Total Career Points', fontsize=16)
plt.xlabel('Total Points')
plt.ylabel('Driver')
plt.show()

### Distribution of Finishing Positions

In [ ]:
results_df['position'] = pd.to_numeric(results_df['position'], errors='coerce')
plt.figure(figsize=(10, 5))
sns.histplot(results_df['position'].dropna(), bins=20, kde=True, color='red')
plt.title('Distribution of F1 Finishing Positions', fontsize=14)
plt.xlabel('Position')
plt.ylabel('Frequency')
plt.show()

## 3. Unstructured Data Analysis (Images with FastAI)
We will apply the same Computer Vision approach used in the `butterfly-image-classifier` notebook to classify F1 Images (e.g. classifying car constructors or driver profiles).

In [ ]:
# Install fastai if not present
!pip install fastai

In [ ]:
from fastai.vision.all import *
from fastai.metrics import error_rate, accuracy
import os

### Check Image Dataset Class Distributions
Assuming images are sorted in folders by class (e.g., `images/Mercedes`, `images/Ferrari`)

In [ ]:
IMAGE_DIR = '../ingestion/images/dataset'

# Since we might not have downloaded the images yet, we mock a safe check
if os.path.exists(IMAGE_DIR):
    labels = os.listdir(IMAGE_DIR)
    counts = {label: len(os.listdir(os.path.join(IMAGE_DIR, label))) for label in labels}
    
    # Plot exactly like the butterfly notebook
    data = dict(sorted(counts.items(), key=lambda item: item[1], reverse=True))
    names = list(data.keys())
    values = list(data.values())

    plt.rcParams["figure.figsize"] = (12,4)
    plt.bar(range(len(names)), values, tick_label=names, color='blue')
    plt.xticks(rotation=45)
    plt.margins(x=0.01)
    plt.title('Number of F1 Images for Each Category')
    plt.box(False)
    plt.show()
else:
    print(f"Directory {IMAGE_DIR} not found. Please place training images in this folder to visualize distributions.")

### FastAI DataBlock and Dataloaders
Create a wrapper around the original data to normalize it for the deep learning model.

In [ ]:
if os.path.exists(IMAGE_DIR):
    block = DataBlock( 
        blocks = (ImageBlock, CategoryBlock), # x = image, y = label
        get_items = get_image_files, # items are images
        splitter = RandomSplitter(valid_pct=0.2, seed=42),
        get_y = parent_label,  # uses folder name as label
        item_tfms = Resize(224), 
        batch_tfms = aug_transforms(max_rotate=20.0, max_zoom=1.2)
    )
    
    # Create a dataloader
    loaders = block.dataloaders(IMAGE_DIR)
    
    # Visualize a batch of images
    loaders.show_batch(max_n=8, nrows=2, ncols=4)
else:
    print("Skipping FastAI DataBlock creation because image directory is empty.")

### Initializing the ResNet50 CNN Model
This sets up a `cnn_learner` ready to be trained, matching the approach in the butterfly model.

In [ ]:
if os.path.exists(IMAGE_DIR):
    learner = cnn_learner(loaders, resnet50, metrics=[error_rate, accuracy])
    
    # To start training, simply run:
    # learner.fine_tune(1)
    # lr_suggestion = learner.lr_find()
    print("ResNet50 Learner Successfully Initialized!")